# End-to-End Wound Segmentation Pipeline
This notebook guides you through setting up and running the UNet + EfficientNet-B4 Wound Segmentation pipeline:
1. **Setup & Installation**
2. **Download Dataset from Kaggle** (`mohsanyaseen/wound-segmentation-yolo-format`)
3. **Verify Dataset Structure**
4. **Model Training** (Phase 1 & Phase 2 fine-tuning with resume option)
5. **Model Evaluation** (Validating on Dice, IoU, Precision, and Recall)
6. **Inference & Visualization** (Predicting and plotting side-by-side)

## 1. Setup & Installation
First, let's install the dependencies needed to build the model, load datasets, run training, and download the dataset from Kaggle.

In [1]:
# Install core training packages
!pip install -q segmentation-models-pytorch albumentations opencv-python matplotlib

# Install Kaggle downloader tools
!pip install -q kaggle opendatasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.6 MB/s eta 0:00:00


## 2. Download Dataset from Kaggle
The dataset is uploaded at `mohsanyaseen/wound-segmentation-yolo-format`. You can download it using either the **Official Kaggle API** or the **`opendatasets` library**.

### Option A: Using the Kaggle API
To download using the official Kaggle API:
1. Go to your **Kaggle Account** page.
2. Click **Create New API Token** to download `kaggle.json`.
3. Put your username and key in the code block below.

In [3]:
import os

# Set your Kaggle API credentials
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"  # Replace with your username
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"            # Replace with your API key

import kaggle

dataset_slug = 'mohsanyaseen/wound-segmentation-yolo-format'
download_path = 'wound_dataset'

print(f"Downloading dataset '{dataset_slug}' to '{download_path}'...")
# Download and unzip the files automatically
kaggle.api.dataset_download_files(dataset_slug, path=download_path, unzip=True)
print("Dataset downloaded and extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/mohsanyaseen/wound-segmentation-yolo-format
Dataset downloaded and extracted successfully!


### Option B: Using `opendatasets`
If you don't want to set environment variables, you can use the `opendatasets` library, which will prompt you interactively for your Kaggle username and key.

In [ ]:
import opendatasets as od
import os
import shutil

# Download using the Kaggle dataset URL
dataset_url = "https://www.kaggle.com/datasets/mohsanyaseen/wound-segmentation-yolo-format"
print("Starting download...")
od.download(dataset_url)

# Move the downloaded folder to "wound_dataset" to match config defaults
src_dir = "wound-segmentation-yolo-format"
dst_dir = "wound_dataset"

if os.path.exists(src_dir) and not os.path.exists(dst_dir):
    shutil.move(src_dir, dst_dir)
    print(f"Moved dataset folder '{src_dir}' to '{dst_dir}'")
elif os.path.exists(dst_dir):
    print(f"Dataset already exists in '{dst_dir}'.")

### Clone Repo

In [4]:
!git clone https://github.com/Mohsan57/Wound-Segmentation-UNet-And-EfficientNet-B4.git

Cloning into 'Wound-Segmentation-UNet-And-EfficientNet-B4'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 90 (delta 48), reused 64 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 139.16 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [5]:
!mv /content/Wound-Segmentation-UNet-And-EfficientNet-B4/* /content/

## 3. Verify Dataset Directory Structure
Let's verify that the dataset images, YOLO labels, and mask files are in place. The training scripts expect:
- `wound_dataset/images/train`
- `wound_dataset/images/val`
- `wound_dataset/labels/train` (optional, can be generated/loaded on-the-fly)
- `wound_dataset/labels/val` (optional)
- `wound_dataset/masks` (pre-made binary mask PNGs matching the image stems)

In [ ]:
from pathlib import Path

data_root = Path("wound_dataset")
print(f"Checking data root: {data_root.resolve()}")
print("Exists:", data_root.exists())

if data_root.exists():
    paths_to_check = [
        "images/train",
        "images/val",
        "labels/train",
        "labels/val",
        "masks"
    ]
    for rel_path in paths_to_check:
        p = data_root / rel_path
        if p.exists():
            count = len(list(p.glob("*")))
            print(f"  ✓ Found '{rel_path}' containing {count} files.")
        else:
            print(f"  ✗ Warning: '{rel_path}' does not exist.")
else:
    print("Dataset directory not found. Please ensure download completed successfully.")

## 4. Run Model Training
The pipeline supports Phase-Aware Fine-Tuning:
- **Phase 1 (Epochs 1-10)**: Encoder is frozen to train the decoder and classification head parameters.
- **Phase 2 (Epoch 11+)**: Encoder is unfrozen for end-to-end training with a smaller learning rate on the encoder.

Other features:
- **Automatic Mixed Precision (AMP)** for fast training.
- **Gradient Accumulation** to handle larger batch sizes.
- **NaN Loss Protection** to halt training if weights begin to diverge.
- **Dynamic GPU Memory Logging** to check hardware utilization.
- **Atomic Checkpoint Saves** to prevent corruption if interrupted.

You can launch the training script from the notebook:

In [ ]:
# To start training from scratch on single GPU or CPU:
!python train.py

# Or to start training on 2 GPUs (Kaggle Multi-GPU):
# !torchrun --nproc_per_node=2 train.py

# Or to resume training from the last automatic checkpoint:
# !python train.py --resume checkpoints/last_model.pth

# Or to resume training from a specific path:
# !python train.py --resume checkpoints/best_model.pth

## 5. Model Evaluation
Evaluate the trained model checkpoint on the validation split. This script loads the specified checkpoint and calculates metrics:
- **Dice Coefficient**
- **Intersection-over-Union (IoU)**
- **Precision**
- **Recall**
- **Loss**

In [ ]:
# Run validation evaluation using the best checkpoint
!python evaluate.py --checkpoint drive/MyDrive/unetplus_mobilenet.pth

usage: evaluate.py [-h] [--checkpoint CHECKPOINT] [--output_dir OUTPUT_DIR]
                   [--device DEVICE]
evaluate.py: error: unrecognized arguments: --export onnx


In [3]:
!unrar x drive/MyDrive/unseen_data.rar -d unseen_data


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from drive/MyDrive/unseen_data.rar

Creating    unseen_data                                               OK
Extracting  unseen_data/processed_42463520.png                           0  OK 
Extracting  unseen_data/processed_42795613.png                           0  OK 
Extracting  unseen_data/processed_42925587.png                           0  OK 
Extracting  unseen_data/processed_42960140.png                           0  OK 
Extracting  unseen_data/processed_43079279.png                           1  OK 
Extracting  unseen_data/processed_43189696.png                           1  OK 
Extracting  unseen_data/processed_43311605.png                           1  OK 
Extracting  unseen_data/processed_43311685.png                           1  OK 
Extracting  unseen_data/processed_43752579.png                           2  OK 
Extracting  unseen_data/processed_43769828.png                           2  OK 
Extr

In [11]:
!pip install onnx2tf tensorflow onnxscript

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 80.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 136.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 96.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 98.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━

In [12]:
!python inference.py --checkpoint drive/MyDrive/unetplus_mobilenet.pth --export onnx

/content/model.py:88: UserWarning: [Warning] Attention type 'scse' is enabled with lightweight mobile backbone 'mobilenet_v2'. Global pooling or fully connected attention layers can bottleneck execution on mobile NPUs/GPUs. Consider setting decoder_attention_type=None or using mobile_mode=True.
  warnings.warn(
[Model] Architecture  : unetplusplus + mobilenet_v2  (decoder attention: scse)
[Model] Total params  : 7,103,094
[Model] Trainable     : 7,103,094
[Checkpoint] Loading <- drive/MyDrive/unetplus_mobilenet.pth  (device: cpu)
[Checkpoint] Loaded <- drive/MyDrive/unetplus_mobilenet.pth  (epoch 53)
/content/inference.py:245: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0611 08:30:09.275000 17514 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because t

## 6. Inference and Visualization
Let's select a validation image, perform inference using our best checkpoint, and plot the original image, ground truth mask, and predicted mask side-by-side to visually inspect the segmentation quality.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from config import Config
from model import build_model
from dataset import get_val_transforms

# Initialize configuration
cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Build and load the model
model = build_model(
    architecture=cfg.architecture,
    encoder_name=cfg.encoder_name,
    encoder_weights=None,  # Do not download imagenet weights for evaluation
    in_channels=cfg.image_channels,
    num_classes=cfg.num_classes,
).to(device)

checkpoint_path = Path(cfg.checkpoint_dir) / "best_model.pth"
if checkpoint_path.exists():
    print(f"Loading checkpoint weights from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    # Support both full checkpoints and weights-only dictionaries
    state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    print("Checkpoint loaded successfully!")
else:
    print(f"Warning: Checkpoint not found at '{checkpoint_path}'. Prediction will be random.")

model.eval()

# 2. Grab a validation image and mask
val_images_dir = Path(cfg.data_root) / "images" / "val"
val_masks_dir = Path(cfg.data_root) / "masks"

if not val_images_dir.exists():
    print(f"Validation images directory '{val_images_dir}' not found.")
else:
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_files = sorted([p for p in val_images_dir.iterdir() if p.suffix.lower() in valid_exts])
    
    if len(image_files) == 0:
        print("No validation images found.")
    else:
        # Choose the first validation image for display
        test_img_path = image_files[0]
        print(f"Displaying predictions for: {test_img_path.name}")
        
        # Read original image (BGR to RGB)
        img_bgr = cv2.imread(str(test_img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]
        
        # Preprocess and prepare tensor
        transform = get_val_transforms(cfg.image_size)
        augmented = transform(image=img_rgb)
        input_tensor = augmented["image"].unsqueeze(0).to(device)
        
        # Predict mask
        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.sigmoid(logits)
            pred_mask = (probs > cfg.threshold).cpu().squeeze().numpy().astype(np.uint8)
            
        # Resize predicted mask back to original resolution
        pred_mask_resized = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)
        
        # Load ground truth mask if available
        gt_mask = np.zeros((h, w), dtype=np.uint8)
        mask_file = val_masks_dir / f"{test_img_path.stem}.png"
        if mask_file.exists():
            gt_mask_bgr = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
            gt_mask = (gt_mask_bgr > 127).astype(np.uint8)
        else:
            print("Ground truth mask file not found in 'masks' directory.")
            
        # 3. Plot side-by-side
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original Image
        axes[0].imshow(img_rgb)
        axes[0].set_title(f"Original Image ({w}x{h})")
        axes[0].axis("off")
        
        # Ground Truth
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("Ground Truth Mask")
        axes[1].axis("off")
        
        # Prediction
        axes[2].imshow(pred_mask_resized, cmap="gray")
        axes[2].set_title("Predicted Mask (Sigmoid > 0.5)")
        axes[2].axis("off")
        
        plt.tight_layout()
        plt.show()